In [ ]:
train_data_location = "../data/train"

In [7]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Standard preprocessing for MobileNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), # Data Augmentation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_data = datasets.ImageFolder(train_data_location, transform=transform)
print(train_data.class_to_idx)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

{'overripe': 0, 'ripe': 1}


In [8]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# Load pre-trained model
model = models.mobilenet_v2(weights='DEFAULT')

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Replace the classifier (last layer)
# MobileNetV2's classifier input is 1280
model.classifier[1] = nn.Linear(1280, 2) 

model = model.to(device)

In [10]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

def train_one_epoch():
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()   # Reset gradients
        outputs = model(inputs) # Forward pass
        loss = criterion(outputs, labels)
        loss.backward()         # Backward pass (autograd)
        optimizer.step()        # Update weights

        running_loss += loss.item()
    return running_loss / len(train_loader)

for epoch in range(10):
    loss = train_one_epoch()
    print(f"Epoch {epoch+1}, Training Loss: {loss:.4f}")

# After training:
torch.save(model.state_dict(), '../saved_models/MobileNet.pth')

Epoch 1, Training Loss: 0.2205
Epoch 2, Training Loss: 0.1958
Epoch 3, Training Loss: 0.1723
Epoch 4, Training Loss: 0.1465
Epoch 5, Training Loss: 0.1337
Epoch 6, Training Loss: 0.1149
Epoch 7, Training Loss: 0.1150
Epoch 8, Training Loss: 0.0838
Epoch 9, Training Loss: 0.0693
Epoch 10, Training Loss: 0.0639
